# Chapter 14 — Self-Play & Constitutional AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part3_advanced/14_selfplay_constitutional_ai.ipynb)

**Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar (Packt)

Three techniques for self-improvement without human labellers:

1. **Constitutional AI (CAI)** — critique → revise loop guided by explicit principles
2. **STaR (Self-Taught Reasoner)** — collect correct reasoning traces + rationalise wrong ones
3. **SPIN (Self-Play Fine-Tuning)** — model competes against its own earlier version

*Previous chapter: test-time compute. Next: multi-objective RL and agentic systems.*

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers==4.40.0 torch accelerate

## 1. Imports

In [ ]:
import re
import copy
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
random.seed(11)
np.random.seed(11)
torch.manual_seed(11)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Load distilgpt2

In [ ]:
MODEL_ID = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()
print(f'Loaded {MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

## 3. Constitutional AI Pipeline

### 3a. The Constitution

A **constitution** is a set of human-authored principles. Instead of labelling each response, we ask the model to judge its own outputs against these principles.

In [ ]:
CONSTITUTION = [
    'Responses should be helpful and informative.',
    'Responses should not encourage harmful or dangerous activities.',
    'Responses should be honest and avoid making up facts.',
    'Responses should be respectful to all people.',
    'Responses should be concise and avoid unnecessary padding.',
]

print('Constitution:')
for i, p in enumerate(CONSTITUTION, 1):
    print(f'  {i}. {p}')

### 3b. Generate, Critique, Revise

In [ ]:
def gen(prompt: str, max_new: int = 60, temp: float = 0.7) -> str:
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=200).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, temperature=temp,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    new = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()


def generate_initial_response(user_prompt: str) -> str:
    return gen(f'User: {user_prompt}\nAssistant:', max_new=60)


def generate_critique(response: str, principle: str) -> str:
    prompt = (
        f'Principle: {principle}\n'
        f'Response: {response[:120]}\n'
        f'Critique (does the response follow the principle?):'
    )
    return gen(prompt, max_new=40, temp=0.6)


def generate_revision(response: str, critique: str) -> str:
    prompt = (
        f'Original: {response[:100]}\n'
        f'Issue: {critique[:80]}\n'
        f'Improved response:'
    )
    return gen(prompt, max_new=60, temp=0.7)


print('CAI pipeline functions defined.')

### 3c. Full CAI Loop on 5 Prompts

In [ ]:
USER_PROMPTS = [
    'How do I get better at coding?',
    'What is the fastest way to lose weight?',
    'Tell me how to pick a lock.',
    'What causes climate change?',
    'How do I make money quickly?',
]

cai_sft_pairs = []  # will be used for fine-tuning

for prompt in USER_PROMPTS:
    print(f'\nPrompt: {prompt}')
    initial = generate_initial_response(prompt)
    print(f'  [Initial]  {initial[:90]}')

    # Apply each constitutional principle in sequence
    current = initial
    for principle in CONSTITUTION[:2]:  # 2 principles per prompt for speed
        critique = generate_critique(current, principle)
        revised  = generate_revision(current, critique)
        current  = revised

    print(f'  [Revised]  {current[:90]}')
    cai_sft_pairs.append({'prompt': prompt, 'initial': initial, 'revised': current})

print(f'\nCollected {len(cai_sft_pairs)} SFT pairs from CAI.')

## 4. STaR — Self-Taught Reasoner

**STaR** builds a training set from the model's own outputs:
- If the model answers **correctly** → keep the reasoning trace as-is
- If the model answers **wrongly** → generate a *rationalisation*: "Given that the answer is X, show how to get there"

Both successes and rationalisations are used to fine-tune the model.

In [ ]:
ARITH_PROBLEMS = [
    ('What is 3 + 4?', 7), ('What is 10 - 3?', 7), ('What is 5 + 6?', 11),
    ('What is 8 - 2?', 6), ('What is 4 + 9?', 13), ('What is 15 - 7?', 8),
    ('What is 2 + 11?', 13), ('What is 20 - 5?', 15), ('What is 7 + 8?', 15),
    ('What is 9 - 4?', 5),
]


def extract_answer(text: str):
    for pat in [r'(?:answer|=)\s*(-?\d+\.?\d*)', r'(-?\d+\.?\d*)\s*$', r'(-?\d+\.?\d*)',]:
        m = re.search(pat, text.strip(), re.IGNORECASE)
        if m:
            try: return float(m.group(1))
            except ValueError: pass
    return None


def generate_rationalisation(question: str, gold: float) -> str:
    """Given the correct answer, ask the model to show how to derive it."""
    prompt = f'Q: {question}\nThe correct answer is {int(gold)}. Show step-by-step reasoning:\n'
    return gen(prompt, max_new=60, temp=0.7)


star_dataset = []
successes, rationalisations = 0, 0

for question, gold in ARITH_PROBLEMS:
    prompt = f'Q: {question}\nLet\'s think step by step.\n'
    resp = gen(prompt, max_new=50, temp=0.8)
    pred = extract_answer(resp)
    correct = (pred is not None and abs(pred - gold) <= 0.5)

    if correct:
        # Keep successful reasoning trace
        star_dataset.append({'question': question, 'reasoning': resp, 'source': 'success'})
        successes += 1
    else:
        # Rationalise: given the answer, show the reasoning
        rational = generate_rationalisation(question, gold)
        star_dataset.append({'question': question, 'reasoning': rational, 'source': 'rationalised'})
        rationalisations += 1

print(f'STaR dataset: {len(star_dataset)} items  ({successes} successes, {rationalisations} rationalisations)')
for item in star_dataset[:3]:
    print(f'  [{item["source"]:12s}] {item["question"]:25s} -> {item["reasoning"][:50]}')

### 4a. Fine-tune on STaR dataset

In [ ]:
def sft_on_dataset(mdl, items: list, epochs: int = 3, lr: float = 5e-5) -> list:
    """SFT on question+reasoning pairs."""
    opt = torch.optim.AdamW(mdl.parameters(), lr=lr)
    mdl.train()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for item in items:
            text = f"Q: {item['question']}\nA: {item['reasoning']}"
            enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(DEVICE)
            opt.zero_grad()
            out = mdl(**enc, labels=enc['input_ids'])
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step()
            epoch_loss += out.loss.item()
        avg = epoch_loss / len(items)
        losses.append(avg)
        print(f'  Epoch {epoch+1}/{epochs}  loss={avg:.4f}')
    return losses


def accuracy_on_problems(mdl, problems: list, n_tries: int = 2) -> float:
    """Fraction of problems solved correctly (best-of-n_tries)."""
    mdl.eval()
    hits = 0
    for question, gold in problems:
        prompt = f'Q: {question}\nLet\'s think step by step.\n'
        for _ in range(n_tries):
            resp = gen(prompt, max_new=40, temp=0.7)
            if extract_answer(resp) is not None and abs(extract_answer(resp) - gold) <= 0.5:
                hits += 1; break
    return hits / len(problems)


acc_before = accuracy_on_problems(model, ARITH_PROBLEMS)
print(f'Accuracy BEFORE STaR fine-tuning: {acc_before:.2%}')

sft_on_dataset(model, star_dataset, epochs=3)

acc_after = accuracy_on_problems(model, ARITH_PROBLEMS)
print(f'Accuracy AFTER STaR fine-tuning:  {acc_after:.2%}')

## 5. SPIN — Self-Play Fine-Tuning

**SPIN** treats the model's previous checkpoint as the "opponent":
- Generate responses with current model → score them → highest = **chosen**, lowest = **rejected**
- Apply a **DPO-style update** so the model prefers its own best outputs over its own worst

This is a self-play loop: each iteration the model improves against its own previous version.

In [ ]:
SPIN_PROMPTS = [
    'Explain why the sky is blue in one sentence.',
    'What is the capital of France?',
    'How do plants make food?',
    'Name the planets in our solar system.',
    'What is 2 + 2?',
    'Define the word entropy.',
    'What is the speed of light?',
    'How many continents are there?',
    'What element has atomic number 1?',
    'What is the Pythagorean theorem?',
]


def simple_quality(response: str) -> float:
    """Heuristic quality score: length, no repetition, sentence structure."""
    words = response.split()
    if len(words) < 3:
        return 0.0
    unique_ratio = len(set(words)) / len(words)
    length_score = min(len(words) / 30, 1.0)
    return 0.6 * unique_ratio + 0.4 * length_score


def dpo_loss_pair(mdl, ref_mdl, prompt: str, chosen: str, rejected: str,
                  beta: float = 0.1) -> torch.Tensor:
    """DPO loss for one (chosen, rejected) pair."""
    def log_prob(m, text):
        ids = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).input_ids.to(DEVICE)
        if ids.shape[1] < 2:
            return torch.tensor(0.0)
        with torch.set_grad_enabled(m is mdl):
            logits = m(ids).logits
        lp = F.log_softmax(logits[:, :-1, :], dim=-1)
        return lp.gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1).mean()

    lp_chosen_pi  = log_prob(mdl,     prompt + chosen)
    lp_chosen_ref = log_prob(ref_mdl, prompt + chosen).detach()
    lp_reject_pi  = log_prob(mdl,     prompt + rejected)
    lp_reject_ref = log_prob(ref_mdl, prompt + rejected).detach()

    log_ratio = (lp_chosen_pi - lp_chosen_ref) - (lp_reject_pi - lp_reject_ref)
    return -F.logsigmoid(beta * log_ratio)


print('SPIN components defined.')

### 5a. SPIN Training Loop (3 Iterations)

In [ ]:
N_SPIN_ITERS = 3
N_CANDIDATES = 4
spin_rewards = []  # track implicit reward (margin) per iteration

for spin_iter in range(N_SPIN_ITERS):
    # Snapshot the current model as the reference (opponent)
    ref_snap = copy.deepcopy(model)
    ref_snap.eval()
    for p in ref_snap.parameters():
        p.requires_grad_(False)

    opt = torch.optim.AdamW(model.parameters(), lr=5e-6)
    iter_rewards = []

    for prompt_text in SPIN_PROMPTS:
        # Generate N_CANDIDATES responses
        model.eval()
        cands = [gen(prompt_text, max_new=50, temp=0.9) for _ in range(N_CANDIDATES)]
        scores = [simple_quality(c) for c in cands]

        best_idx  = int(np.argmax(scores))
        worst_idx = int(np.argmin(scores))
        chosen   = cands[best_idx]
        rejected = cands[worst_idx]
        margin = scores[best_idx] - scores[worst_idx]
        iter_rewards.append(margin)

        if scores[best_idx] == scores[worst_idx]:
            continue  # no signal

        # DPO update
        model.train()
        opt.zero_grad()
        loss = dpo_loss_pair(model, ref_snap, prompt_text, chosen, rejected)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

    avg_margin = float(np.mean(iter_rewards))
    spin_rewards.append(avg_margin)
    print(f'SPIN iter {spin_iter+1}/{N_SPIN_ITERS}  avg_margin={avg_margin:.4f}')

### 5b. Implicit Reward Over SPIN Iterations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# SPIN reward
axes[0].plot(range(1, N_SPIN_ITERS+1), spin_rewards, 'b-o', ms=8)
axes[0].set_xlabel('SPIN iteration')
axes[0].set_ylabel('Avg quality margin (chosen − rejected)')
axes[0].set_title('SPIN: Implicit Reward Over Iterations')
axes[0].set_xticks(range(1, N_SPIN_ITERS+1))
axes[0].grid(True, alpha=0.3)

# STaR before/after bar chart
axes[1].bar(['Before STaR', 'After STaR'], [acc_before, acc_after],
            color=['#cccccc', '#4a90d9'])
axes[1].set_ylim(0, 1.0)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('STaR: Accuracy Before and After Fine-Tuning')
for i, v in enumerate([acc_before, acc_after]):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('selfplay_cai_summary.png', dpi=120)
plt.show()

## 6. Summary

| Method | Human labels needed? | Key mechanism | Use case |
|---|---|---|---|
| **CAI** | Only the constitution | Critique + revise guided by principles | Safety, helpfulness alignment |
| **STaR** | Known answers only | Collect correct traces + rationalise failures | Arithmetic, reasoning tasks |
| **SPIN** | None | Self-play DPO against previous checkpoint | General instruction following |

All three generate training signal **without human preference labels** — a key advantage when labels are expensive.

**Chapter 15** extends to multi-objective RL: when we must balance helpfulness, harmlessness, and honesty simultaneously.